# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"ID: {metadata.id}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Authors: {[author['@id'] for author in getattr(metadata, 'author', [])]}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets available by their @id
print("Available Record Sets (@id):")
record_sets = dataset.list_record_sets()
for i, rec in enumerate(record_sets):
    print(f"{i+1}. @id: {rec['@id']}  |  name: {rec.get('name', '')}")

# Get fields for each record set (by @id)
record_set_fields = {}
for rec in record_sets:
    fields = dataset.list_fields(record_set=rec['@id'])
    print(f"\nFields for record set '@id': {rec['@id']}")
    for f in fields:
        print(f"- @id: {f['@id']:<40} | name: {f.get('name','')} | dataType: {f.get('dataType','')}")
    record_set_fields[rec['@id']] = fields

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Choose record set(s) for extraction -- enter their @id(s) from above
record_set_ids = [rec['@id'] for rec in record_sets]
dataframes = {}

for rec_id in record_set_ids:
    print(f"\nExtracting records from record set: {rec_id}")
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded DataFrame with columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {rec_id}. Error: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate with the first available record set.

In [ ]:
import numpy as np

# Choose the first record set for analysis
if len(record_set_ids) > 0:
    selected_record_set = record_set_ids[0]
    df = dataframes[selected_record_set]
    print(f"\nWorking with record set: {selected_record_set}")

    # List numeric fields by inspecting data types in the sample
    numeric_field = None
    for col in df.columns:
        # Try numeric conversion on a sample of values to infer if a field is numeric
        sample_nonnulls = df[col].dropna().astype(str).sample(min(10, len(df[col].dropna())), random_state=0)
        try:
            _ = pd.to_numeric(sample_nonnulls)
            numeric_field = col
            break
        except:
            continue

    if numeric_field is not None:
        print(f"Selected numeric field for demonstration: {numeric_field}")
        # Convert column to numeric (will introduce NaN for missing values)
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt group by a categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object':
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename(columns={numeric_field: f"mean_{numeric_field}"})
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found in this record set for demonstration.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_set_ids) > 0 and numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}' in record set '{selected_record_set}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to discover and explore a Croissant FAIR² dataset using the `mlcroissant` Python library.

- We loaded metadata and identified available record sets and their fields using only `@id` references.
- We extracted record set data programmatically and carried out initial exploratory data analysis, including filtering and normalization.
- We plotted basic visualizations if numeric fields were available.

For advanced usage, consult the official [`mlcroissant` documentation](https://mlcommons.github.io/croissant/api/python/) or examine additional record sets and their fields.